# Hot-Seat Meter: walk-forward backtest (step H3b)

Thin notebook: it only reads the outputs of `uv run twm hotseat backtest --labels verified|suggested`
(PROJECT_SPEC 8.5; label rules and model choices in `docs/hot_seat.md`). It reads
`reports/hot_seat/` when a verified run exists, otherwise the **provisional** run in
`data/hot_seat/provisional/` (research-prefilled labels, not verified by the owner: do not quote
those numbers).

In [ ]:
import io
import os

import polars as pl
from IPython.display import Image, Markdown, display
from matplotlib.figure import Figure

from twm.config import ROOT
from twm.modules.hot_seat import backtest_report as hr

os.chdir(ROOT)  # every path below is relative to the project folder
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_rows(40)
pl.Config.set_float_precision(3)
VERIFIED, PROVISIONAL = ROOT / "reports/hot_seat", ROOT / "data/hot_seat/provisional"
OUT = VERIFIED if (VERIFIED / hr.FILES["metrics"]).exists() else PROVISIONAL
if OUT == PROVISIONAL:
    display(Markdown(hr.PROVISIONAL))


def read(key):
    return pl.read_csv(OUT / hr.FILES[key])


metrics, firings, coefs = read("metrics"), read("firings"), read("coefficients")
reliability, groups = read("reliability"), read("groups")
print(f"outputs: {OUT.relative_to(ROOT)}")

## Firings per season

Coach-seasons with a positive departure (interims apart).

In [ ]:
firings

## Headline: all rows, week 12, end of season

Main run, the models' own probabilities, 95% season-block intervals.

In [ ]:
head = metrics.filter(
    (pl.col("variant") == "main")
    & (pl.col("prob") == "prob")
    & pl.col("slice").is_in(["all", "week_12", "end_of_season"])
)
head.select("model", "slice", "metric", "value", "lo", "hi", "n_rows", "n_pos").sort(
    "slice", "metric", "model"
)

## ROC-AUC and PR-AUC by as-of week

In [ ]:
BLUE, ORANGE, GREY, INK, GRID, SURFACE = (
    "#2a78d6",
    "#d9822b",
    "#8a8984",
    "#52514e",
    "#e1e0d9",
    "#fcfcfb",
)
COLORS = {"logit": BLUE, "hazard": ORANGE, "base_win_pct": GREY}


def show(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=72, facecolor=SURFACE)
    display(Image(buf.getvalue()))


def style(ax):
    ax.set_facecolor(SURFACE)
    ax.grid(color=GRID, linewidth=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.tick_params(colors=INK, labelsize=9)


weekly = metrics.filter(
    (pl.col("variant") == "main")
    & (pl.col("prob") == "prob")
    & pl.col("slice").str.starts_with("week_")
).with_columns(pl.col("slice").str.slice(5).cast(pl.Int32).alias("week"))
fig = Figure(figsize=(10, 3.8), layout="constrained", facecolor=SURFACE)
for ax, metric in zip(fig.subplots(1, 2), ("roc_auc", "pr_auc"), strict=True):
    for model, color in COLORS.items():
        d = weekly.filter((pl.col("model") == model) & (pl.col("metric") == metric)).sort("week")
        ax.plot(d["week"], d["value"], color=color, label=model, linewidth=1.8)
        ax.fill_between(d["week"], d["lo"], d["hi"], color=color, alpha=0.12, linewidth=0)
    style(ax)
    ax.set_title(metric.replace("_", "-").upper(), loc="left", fontsize=11)
    ax.set_xlabel("as-of week (Tuesday after the week's games)", color=INK)
fig.axes[0].legend(frameon=False, fontsize=9)
show(fig)

## Reliability at season end

Predicted vs observed, 10-point bins (own probabilities).

In [ ]:
eos = reliability.filter(
    (pl.col("slice") == "end_of_season") & (pl.col("prob") == "prob") & (pl.col("n") > 0)
)
fig = Figure(figsize=(4.8, 4.2), layout="constrained", facecolor=SURFACE)
ax = fig.subplots()
ax.plot([0, 1], [0, 1], color=GRID, linewidth=1)
for model in ("logit", "hazard"):
    d = eos.filter(pl.col("model") == model)
    ax.plot(d["mean_pred"], d["observed"], marker="o", color=COLORS[model], label=model)
style(ax)
ax.set_xlabel("mean predicted risk", color=INK)
ax.set_ylabel("observed share fired", color=INK)
ax.legend(frameon=False, fontsize=9)
show(fig)

## Interims and censored rows, coefficients

In [ ]:
groups.filter(pl.col("variant") == "main")

In [ ]:
coefs.filter(pl.col("variant") == "main").sort(
    "model", pl.col("last_fold").abs(), descending=[False, True]
)